# Keystone Eval Comparison — 2026-02-25 (before) vs 2026-02-27 (after codex fix)

Comparing the effect of the AGENTS.md + short prompt fix for codex-mini.

- **Before (2026-02-25):** codex-mini got full 13k prompt → 0% pass rate
- **After (2026-02-27):** codex-mini gets short prompt + AGENTS.md → should improve

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def load_eval_results(base_dir: Path, run_label: str) -> list[dict]:
    records = []
    for p in base_dir.rglob("eval_result.json"):
        with p.open() as f:
            d = json.load(f)
        rel = p.relative_to(base_dir)
        model = rel.parts[0]
        repo = rel.parts[1]
        trial = int(rel.parts[2].replace("trial_", ""))
        br = d.get("bootstrap_result") or {}
        ver = br.get("verification") or {}
        agent = br.get("agent") or {}
        cost = agent.get("cost") or {}
        repo_entry = d.get("repo_entry") or {}
        records.append(
            {
                "run": run_label,
                "model": model,
                "repo": repo,
                "trial": trial,
                "success": d.get("success", False),
                "bootstrap_success": br.get("success", False),
                "verification_success": ver.get("success", False),
                "error_message": br.get("error_message", ""),
                "image_build_seconds": ver.get("image_build_seconds"),
                "test_execution_seconds": ver.get("test_execution_seconds"),
                "agent_duration_seconds": agent.get("duration_seconds"),
                "agent_timed_out": agent.get("timed_out", False),
                "cost_usd": cost.get("cost_usd"),
                "input_tokens": cost.get("token_spending", {}).get("input"),
                "output_tokens": cost.get("token_spending", {}).get("output"),
                "language": repo_entry.get("language", ""),
                "difficulty": repo_entry.get("difficulty", ""),
            }
        )
    return records


HOME = Path.home()
before = load_eval_results(HOME / "keystone_evals" / "2026-02-25_thad", "before (02-25)")
after = load_eval_results(HOME / "keystone_evals" / "2026-02-27", "after (02-27)")

df = pd.DataFrame(before + after)
print(f"Loaded {len(df)} total results: {df['run'].value_counts().to_dict()}")
df.head()

## Overall Success Rate: Before vs After

In [ ]:
summary = (
    df.groupby(["run", "model"])
    .agg(
        total=("success", "count"),
        successes=("success", "sum"),
        success_rate=("success", "mean"),
        mean_cost_usd=("cost_usd", "mean"),
        mean_agent_duration=("agent_duration_seconds", "mean"),
    )
    .reset_index()
)

summary["success_rate_pct"] = (summary["success_rate"] * 100).round(1)
summary["mean_cost_usd"] = summary["mean_cost_usd"].round(4)
summary["mean_agent_duration"] = summary["mean_agent_duration"].round(1)
summary[["run", "model", "total", "successes", "success_rate_pct", "mean_cost_usd", "mean_agent_duration"]]

In [ ]:
fig = px.bar(
    summary,
    x="model",
    y="success_rate_pct",
    color="run",
    barmode="group",
    text="success_rate_pct",
    title="Success Rate by Model: Before vs After Codex Fix",
    labels={"success_rate_pct": "Success Rate (%)", "model": "Model"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 50])
fig.show()

## Codex-Mini Deep Dive: Before vs After

In [ ]:
codex_mini = df[df["model"] == "codex-mini-gpt-5.1"].copy()

pivot = codex_mini.pivot_table(index="repo", columns="run", values="success", aggfunc="any")
pivot = pivot.fillna(False)

# Highlight changes
pivot["change"] = pivot.apply(
    lambda r: "NEW PASS" if r.get("after (02-27)", False) and not r.get("before (02-25)", False)
    else ("REGRESSION" if r.get("before (02-25)", False) and not r.get("after (02-27)", False)
    else ("BOTH PASS" if r.get("after (02-27)", False) and r.get("before (02-25)", False)
    else "BOTH FAIL")), axis=1
)

print("codex-mini-gpt-5.1 changes:")
print(pivot["change"].value_counts())
print()
print("Newly passing repos:")
new_passes = pivot[pivot["change"] == "NEW PASS"].index.tolist()
print(new_passes if new_passes else "(none)")
print()
print("Regressions:")
regressions = pivot[pivot["change"] == "REGRESSION"].index.tolist()
print(regressions if regressions else "(none)")

## Per-Repo Heatmap: After (02-27)

In [ ]:
after_df = df[df["run"] == "after (02-27)"]
pivot_after = after_df.pivot_table(index="repo", columns="model", values="success", aggfunc="mean")
pivot_after = pivot_after.sort_index()

fig = px.imshow(
    pivot_after,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=1,
    title="Success by Repo x Model (After Fix — 02-27)",
    labels={"color": "Success Rate"},
    aspect="auto",
    height=max(400, len(pivot_after) * 22),
)
fig.show()

## Success Rate by Difficulty: After (02-27)

In [ ]:
after_df = df[df["run"] == "after (02-27)"]

model_order = (
    after_df.groupby("model")["success"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

diff_df = after_df.groupby(["difficulty", "model"])["success"].mean().reset_index()
diff_df["success"] = (diff_df["success"] * 100).round(1)

fig = px.bar(
    diff_df,
    x="difficulty",
    y="success",
    color="model",
    barmode="group",
    text="success",
    title="Success Rate by Difficulty & Model (After Fix)",
    labels={"success": "Success Rate (%)", "difficulty": "Difficulty"},
    category_orders={"model": model_order},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(yaxis_range=[0, 105])
fig.show()

## Cost & Duration Comparison

In [ ]:
cost_summary = (
    df.groupby(["run", "model"])
    .agg(
        mean_cost=("cost_usd", "mean"),
        median_cost=("cost_usd", "median"),
        mean_duration_min=("agent_duration_seconds", lambda x: (x / 60).mean()),
        median_duration_min=("agent_duration_seconds", lambda x: (x / 60).median()),
        timeouts=("agent_timed_out", "sum"),
    )
    .round(3)
)
cost_summary

## Failure Reasons: Before vs After

In [ ]:
def categorize_error(msg):
    if not msg:
        return "Unknown"
    m = msg.lower()
    if "dockerfile not found" in m:
        return "No files created"
    if "timeout" in m or "timed out" in m:
        return "Agent timeout"
    if "container id" in m and ("finished" in m or "status=" in m):
        return "Modal container crash"
    if "no container with id" in m:
        return "Modal container not found"
    if "build failed" in m:
        return "Docker build failed"
    if "test run failed" in m or ("test" in m and "return code" in m):
        return "Tests failed"
    if any(x in m for x in ["nodename nor servname", "file descriptor not found", "errno", "eof"]):
        return "Infrastructure error"
    return "Other"


failures = df[~df["success"]].copy()
failures["error_category"] = failures["error_message"].apply(categorize_error)

# Focus on codex-mini before vs after
codex_failures = failures[failures["model"] == "codex-mini-gpt-5.1"]
err_df = codex_failures.groupby(["run", "error_category"]).size().reset_index(name="count")

fig = px.bar(
    err_df,
    x="run",
    y="count",
    color="error_category",
    title="Codex-Mini Failure Reasons: Before vs After",
    labels={"count": "Count", "error_category": "Error Category"},
)
fig.show()

## Head-to-Head: Uniquely Solved Repos (After)

In [ ]:
after_df = df[df["run"] == "after (02-27)"]
repo_success = after_df.groupby(["repo", "model"])["success"].any().unstack(fill_value=False)

for model in repo_success.columns:
    others = [c for c in repo_success.columns if c != model]
    unique = repo_success[model] & ~repo_success[others].any(axis=1)
    unique_repos = unique[unique].index.tolist()
    if unique_repos:
        print(f"{model} uniquely solves ({len(unique_repos)}): {', '.join(unique_repos)}")
    else:
        print(f"{model}: no uniquely solved repos")